# InnerSource Acceptance Example <a name="top"></a>

## Table of Contents
1. [Introduction](#introduction)
2. [End-to-end Pinning Example and Explanations](#example)
    - [Libraries and helper functions](#libraries)


## Introduction <a name="introduction"></a>
This stage governs whether the file as a whole or individual records are accepted or rejected.

The Acceptance stage accepts input files from the previous stage can be accepted or rejected based on incoming file quality scores and custom rules. Files are then sharded into individual records (known as events.) Each event maintains knowledge of its parent file as well as quality scores for the parent file and any other derived attributes. The Acceptance stage outputs events (again split into acceptance/accepted or accepted/rejected directories.)

## Input
The input for the acceptance process are validated events, in the default directory or a custom directory.


[Return to Top](#top)

## End-to-end Acceptance Example and Explanations <a name="example"></a>
The Acceptance job orchestrates through the following steps:

1. [**Prepare input events**](#inputevents)
3. [**Run acceptance**](#runacceptance)
4. [**Review acceptance outputs**](#output)


#### Libraries and helper functions <a name="libraries"></a>
First, we begin by importing necessary Python libraries and helper functions. These aid in extracting key information from inputs, intermediary results, and final outputs.

In [1]:
import os
import re
import json
import pandas as pd
pd.set_option('display.max_colwidth', None)

def print_jsonl_1stline(file):
    with open(file, 'r') as f:
        line = f.readline()
        data = json.loads(line)
        print(json.dumps(data, indent=2))

def filter_files(files, substring):
    for file in files:
        if substring in file:
            return file

def print_jsonl_by_eventId(file, eventId):
    with open(file, 'r') as f:
        for line in f:
            data = json.loads(line)
            if data.get("eventId") == eventId:
                print(json.dumps(data, indent=2))
                break

[Return to Top](#top)

#### Local paths <a name="paths"></a>
Subsequently, please revise the path variables to ensure they correctly point to the respective paths of the input, output, and identifying attributes configuration file on your machine.

In [8]:
input_folder = 'example/test-workspace/validation/validated'
output_root_path = 'example/test-workspace/accepted'

[Return to Top](#top)

### 1. Prepare input events <a name="inputevents"></a>
The inputs for the acceptance process are validated events.

Each event is assigned a unique "eventId", with all event details stored as values in various attribute fields.

We read from the inputs and display the first event as an example.

In [13]:
# Delete .DS_Store file 
ds_file_path = os.path.join(input_folder, '.DS_Store')
if os.path.exists(ds_file_path):
    os.remove(ds_file_path)
    print(".DS_Store deleted")
    
# Print the first input event as an example
batch = os.listdir(input_folder)[0]
batch_folder = '/'.join([input_folder, batch])
files = [file for file in os.listdir(batch_folder) if file.endswith('.json')]
file = os.path.join(batch_folder, files[0])
print(f"Batch name: {batch}\n")
print("Sample Validated event:\n")
print_jsonl_1stline(file)

Batch name: E-230001.20161113.T.7104964956669341162

Sample Validated event:

{
  "lineage": {
    "dataSource": "230001",
    "receivedDate": 1485993600000,
    "fileName": "file:/Users/c80844b/Documents/MyWorkspace/apps/RI/oxygen-ri/oxygen-ri-data/E-230001.20161113.T",
    "subscriber": "230001",
    "fileAsOfDate": 1485993600000,
    "datasetId": "7104964956669341162",
    "recordHash": "9a7fac1a698e6bc52b33973d9caba16ec8481a63"
  },
  "eventId": "3140b3bd-df29-6d8d-0000-000000000529",
  "validatedRecordJson": "{\"idType\":1,\"nationalId\":\"00093880780\",\"accountNumber\":\"000000111116107121\",\"fullName\":\"ALBER CLEOFAS RIVAS POTES\",\"situationOfHolder\":0,\"openningDate\":20060503,\"dueDate\":20301231,\"responsibleOrQualityOfDebtor\":0,\"typeOfobligation\":4,\"mortgageSubsidy\":0,\"dateOfSubsidy\":0,\"endOfThatContractCreatesTheObligation\":2,\"wayToPay\":0,\"frequencyOfPayment\":1,\"novelty\":13,\"stateOfOriginAccount\":0,\"stateDateOrigin\":20060503,\"stateOfMind\":5,\"dateS

[Return to Top](#top)

### 2. Run acceptance <a name="runacceptance"></a>
Run acceptance with sample input.

In [17]:
# Uncomment the lines in this block and run acceptance
!cp src/main/resources/log4j.properties /opt/spark/conf/;
!spark-submit --master local  --class com.experian.edf.oxygen.acceptance.app.AcceptanceApp target/oxygen-acceptance-innersource-1.0-jar-with-dependencies.jar env.conf acceptance-application.conf
#--conf "spark.driver.extraJavaOptions=-Dlog4j.configuration=log4j.properties"

23/08/30 17:44:31 INFO TypesafeConfigLoader: Default Config File Used : oxygen-config-default.conf
23/08/30 17:44:31 INFO SparkOxygenAppContext: Config File Used :env.conf
23/08/30 17:44:32 INFO SparkOxygenAppContext: Config File Used :acceptance-application.conf
23/08/30 17:44:32 INFO SparkOxygenAppContext: Config File Used :spark.conf
23/08/30 17:44:32 INFO TypesafeConfigLoader: loading command line parameters if any
23/08/30 17:44:32 INFO TypesafeConfigLoader: {\"application\":{\"batchTableLoadMechanism\":\"hydrator\",\"compression\":\"snappy\",\"configs\":{\"hbase\":{\"bulkLoadDirectory\":\"/Users/c80844b/Documents/InnersourceWorkspace/app/oxygen-ri-app/temp/hbase\",\"namespace\":\"oxygen\",\"namespaceWithEnviromnmentAndGeneration\":\"oxygen:dev_default_person\",\"namespaceWithEnvironment\":\"oxygen:dev\"},\"kafka\":{\"consumer\":{\"auto\":{\"commit\":{\"interval\":{\"ms\":0}},\"offset\":{\"reset\":\"earliest\"}},\"bootstrap\":{\"servers\":\"localhost:9092\"},\"enable\":{\"auto\":{

[Return to Top](#top)

### 3. Review acceptance outputs <a name="output"></a>
The final output of acceptance will divide the accepted and rejected files, based on the creteria defined in the acceptance-application.conf

In [24]:
# Print the first input event as an example
batch = os.listdir(output_root_path)[0]
batch_folder = '/'.join([output_root_path, batch])
files = [file for file in os.listdir(batch_folder) if file.endswith('.json')]
file = os.path.join(batch_folder, files[0])
print(f"Batch name: {batch}\n")
print("Sample Accepted event:\n")
print_jsonl_1stline(file)

NotADirectoryError: [Errno 20] Not a directory: 'example/test-workspace/accepted/.DS_Store'

[Return to Top](#top)

## Conclusion <a name="conclusion"></a>
Pinning is a process that associates events with the same entities. It extracts identity information from the events, reduces the comparison scope from the entire identity universe to small, multidimensional near neighbors for distributed computing, rigorously verifies identity connections within near neighbors through exact matching, and then merges identity connections across near neighbors. Unique pins are then assigned to connected identity clusters to represent individual entities. Ultimately, pins are assigned to the events associated with the connected identities.


[Return to Top](#top)